**MODULE 1 - LOAD AND INSPECT THE MASTER LIST**

Connecting drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Importing required libraries

In [ ]:
import pandas as pd

Starting with initail inspection of plantslist

In [ ]:
plantslist_path = 'plantslist.xlsx'
plantslist = pd.read_excel(plantslist_path)

In [ ]:
print(plantslist.shape)

(474, 4)


In [ ]:
plantslist.head()

,id,name,commonname,lf
0,1,Acacia acinacea s.l.,Gold-dust Wattle,MS
1,2,Acacia aculeatissima,Thin-leaf Wattle,SS
2,3,Acacia dealbata,Silver Wattle,T
3,4,Acacia genistifolia,Spreading Wattle,MS
4,6,Acacia implexa,Lightwood,T


In [ ]:
plantslist.columns.tolist()

['id', 'name', 'commonname', 'lf']

In [ ]:
plantslist.dtypes

,0
id,int64
name,object
commonname,object
lf,object


In [ ]:
plantslist.isna().sum()

,0
id,0
name,0
commonname,0
lf,0


its completely clean file

**MODULE 2 -  Match plants against AusTraits names**

**What we're doing:**

- Loading just the two name columns from traits.csv.
- Normalising both sides (strip whitespace + lowercase).

- **Running the 3-step funnel:**
  - Match against taxon_name only (basic).
  - Match against taxon_name OR original_name (extended - catches synonyms).
  - Export unmatched plants to a CSV for manual review.

In [ ]:
traits_path = 'traits.csv'

Loading only the two name columns

Using latin encosing because some plant names can have latin symbols.

It maps each character to a single byte. The first 128 characters are identical to ASCII, while the remaining 128 include special characters used in Western European languages (e.g., accented letters like ö, é, ñ).

> From google search



In [ ]:
traits_name = pd.read_csv(
    traits_path , usecols = ['taxon_name' , 'original_name'],
    encoding = 'latin-1',
    low_memory=False,
)

Starting from basic instructions

In [ ]:
print(len(traits_name))

1798215


In [ ]:
print(traits_name['taxon_name'].nunique())

33370


normalise names on both sides

In [ ]:
plantslist['name_norm'] = plantslist['name'].str.strip().str.lower()
taxon_norm    = set(traits_name['taxon_name'].dropna().str.strip().str.lower())
original_norm = set(traits_name['original_name'].dropna().str.strip().str.lower())

bsic match - taxon_name or original_name

In [ ]:
plantslist['match_taxon'] = plantslist['name_norm'].isin(taxon_norm)
basic = plantslist['match_taxon'].sum()
print(f"\n[Basic]    Match against taxon_name only:{basic} / {len(plantslist)}")


[Basic]    Match against taxon_name only:324 / 474


In [ ]:
all_known = taxon_norm | original_norm
plantslist['match_extended'] = plantslist['name_norm'].isin(all_known)
extended = plantslist['match_extended'].sum()
print(f"[Extended] Match against taxon_name OR original_name: {extended} / {len(plantslist)}")
print(f"Bonus matches via original_name:+{extended - basic}")

[Extended] Match against taxon_name OR original_name: 415 / 474
Bonus matches via original_name:+91


Exporting the unmatched ones

In [ ]:
unmatched = plantslist[~plantslist['match_extended']]

In [ ]:
print(f"\nUnmatched plants: {len(unmatched)}")
print("\nFirst 30 unmatched:")
for name in unmatched['name'].head(30).tolist():
    print(f"  - {name}")


Unmatched plants: 59

First 30 unmatched:
  - Acacia mucronata ssp. longifolia
  - Acaena  novae-zelandiae
  - Asplenium trichomanes ssp. quadrivalens
  - Astroloma  conostephioides
  - Austrodanthonia  caespitosa
  - Austrodanthonia  duttoniana
  - Austrodanthonia  setacea
  - Austrodanthonia setacea var. setacea
  - Austrostipa  aristiglumis
  - Austrostipa  bigeniculata
  - Austrostipa  elegantissima
  - Austrostipa scabra ssp. falcata
  - Banksia integrifolia ssp. integrifolia
  - Blechnum  cartilagineum
  - Blechnum penna-marina ssp. alpina
  - Brachychiton populneus ssp. populneus
  - Brachyloma  daphnoides
  - Brachyloma ericoides ssp. ericoides
  - Chamaesyce  drummondii
  - Cheilanthes  austrotenuifolia
  - Cheilanthes sieberi ssp. sieberi
  - Chenopodium  nitrariaceum
  - Chysocephalum  apiculatum
  - Convolvulus  erubescens
  - Crassula sieberiana ssp. sieberiana
  - Cymbonotus  preissianus
  - Dianella revoluta s.s.
  - Dillwynia sericea s.l.
  - Disphyma crassifolium ssp.

saving for manual review:

In [ ]:
unmatched[['id', 'name', 'commonname']].to_csv('unmatched_plants.csv', index=False)


In [ ]:
len(unmatched)

59

- Subspecies (ssp.) - ~17 plants. Examples: Asplenium trichomanes ssp. quadrivalens, Banksia integrifolia ssp. integrifolia, Brachychiton populneus ssp. populneus. AusTraits often has data for the species but not the subspecies.

- Varieties (var.) - Austrodanthonia setacea var. setacea. Same problem.

- Sensu lato / stricto (s.l., s.s.) - Dianella revoluta s.s., Dillwynia sericea s.l.. Broad/narrow taxonomic designations.

- Genuinely missing or possibly typo'd - most of the rest (e.g. Astroloma conostephioides, Chysocephalum apiculatum - the second one looks like a typo for Chrysocephalum).

**Rescue pass: strip taxonomic qualifiers**

- For each unmatched plant, strip everything after ssp. / var. / s.l. / s.s. / subsp. / f. / forma.

- Also collapse any double whitespace (defensive).

- Re-match the simplified name against the same all_known set from Module 2.

- Promote rescued plants to "matched" in the main plantslist.

In [ ]:
#using regex
import re

In [ ]:
#stripping taxonomic qualifiers
pattern = re.compile(r'\s+(?:ssp|subsp|var|subvar|forma|f|s\.l|s\.s)\..*$', flags=re.IGNORECASE,)

In [ ]:
def simplify_name(name):
  if not isinstance(name , str):
    return None
  n = name.strip().lower()
  n = re.sub(r'\s+',' ',n)
  n = pattern.sub('', n)
  return n


In [ ]:
print("Sanity check on a few unmatched names:")
sample = [
    "Asplenium trichomanes ssp. quadrivalens",
    "Banksia integrifolia ssp. integrifolia",
    "Austrodanthonia setacea var. setacea",
    "Dianella revoluta s.s.",
    "Dillwynia sericea s.l.",
    "Acaena  novae-zelandiae",   # double-space test
]
for n in sample:
    print(f"  {n!r}  ->  {simplify_name(n)!r}")

Sanity check on a few unmatched names:
  'Asplenium trichomanes ssp. quadrivalens'  ->  'asplenium trichomanes'
  'Banksia integrifolia ssp. integrifolia'  ->  'banksia integrifolia'
  'Austrodanthonia setacea var. setacea'  ->  'austrodanthonia setacea'
  'Dianella revoluta s.s.'  ->  'dianella revoluta'
  'Dillwynia sericea s.l.'  ->  'dillwynia sericea'
  'Acaena  novae-zelandiae'  ->  'acaena novae-zelandiae'


In [ ]:
unmatched = plantslist[~plantslist['match_extended']].copy()
unmatched['name_simplified'] = unmatched['name'].apply(simplify_name)
unmatched['rescued'] = unmatched['name_simplified'].isin(all_known)

rescued       = unmatched[unmatched['rescued']]
still_missing = unmatched[~unmatched['rescued']]

print(f"Rescued via simplified name: {len(rescued)} / {len(unmatched)}")
print(f"Still unmatched after rescue: {len(still_missing)}\n")

print("Rescued plants (original -> matched as):")
for _, row in rescued.iterrows():
    print(f"  - {row['name']:<55} ->  {row['name_simplified']}")

Rescued via simplified name: 53 / 59
Still unmatched after rescue: 6

Rescued plants (original -> matched as):
  - Acacia mucronata ssp. longifolia                        ->  acacia mucronata
  - Acaena  novae-zelandiae                                 ->  acaena novae-zelandiae
  - Asplenium trichomanes ssp. quadrivalens                 ->  asplenium trichomanes
  - Astroloma  conostephioides                              ->  astroloma conostephioides
  - Austrodanthonia  caespitosa                             ->  austrodanthonia caespitosa
  - Austrodanthonia  duttoniana                             ->  austrodanthonia duttoniana
  - Austrodanthonia  setacea                                ->  austrodanthonia setacea
  - Austrodanthonia setacea var. setacea                    ->  austrodanthonia setacea
  - Austrostipa  aristiglumis                               ->  austrostipa aristiglumis
  - Austrostipa  bigeniculata                               ->  austrostipa bigeniculata
  - Austr

In [ ]:
# Update plantslist with rescue results
plantslist['match_strategy'] = 'unmatched'
plantslist.loc[plantslist['match_taxon'], 'match_strategy'] = 'exact_taxon'
plantslist.loc[
    plantslist['match_extended'] & ~plantslist['match_taxon'],
    'match_strategy'
] = 'via_original_name'

rescued_ids = rescued['id'].tolist()
plantslist.loc[plantslist['id'].isin(rescued_ids), 'match_strategy'] = 'species_level_fallback'
plantslist.loc[plantslist['id'].isin(rescued_ids), 'name_norm'] = (
    plantslist.loc[plantslist['id'].isin(rescued_ids), 'name'].apply(simplify_name)
)
plantslist.loc[plantslist['id'].isin(rescued_ids), 'match_extended'] = True

In [ ]:
print("\nFinal match strategy breakdown:")
print(plantslist['match_strategy'].value_counts())
print(f"\nTotal matched: {plantslist['match_extended'].sum()} / {len(plantslist)}")


Final match strategy breakdown:
match_strategy
exact_taxon               324
via_original_name          91
species_level_fallback     53
unmatched                   6
Name: count, dtype: int64

Total matched: 468 / 474


save unmatched plants to a file:

In [ ]:
plantslist[~plantslist['match_extended']][['id', 'name', 'commonname']].to_csv(
    'unmatched_plants.csv', index=False
)

save matched ones

In [ ]:
plantslist.to_csv('plantslist_matched.csv', index=False)
print("Saved plantslist_matched.csv")

Saved plantslist_matched.csv


In [ ]:
print("Final 6 unmatched:")
for name in plantslist[~plantslist['match_extended']]['name'].tolist():
    print(f"  - {name}")

Final 6 unmatched:
  - Chysocephalum  apiculatum
  - Distichlis distichopylla
  - Leptorhyncos  tenuifolius
  - Pimelea microphylla
  - Rumex spp.
  - Selaginella dominii


Hand fix the typo's

> Searched few names on Google



In [ ]:
typo_fixes = [
    # (pattern_to_find, corrected_name, reason)
    ('Chysocephalum',  'Chrysocephalum apiculatum',  "missing 'r' in genus"),
    ('Leptorhyncos',   'Leptorhynchos tenuifolius',  "missing 'h' in genus"),
]

In [ ]:
for pattern, corrected, reason in typo_fixes:
    mask = (~plantslist['match_extended']) & plantslist['name'].str.contains(
        pattern, case=False, na=False
    )
    if not mask.any():
        print(f"  No match for pattern '{pattern}'")
        continue

    original_name = plantslist.loc[mask, 'name'].iloc[0]
    corrected_norm = corrected.strip().lower()
    in_austraits = corrected_norm in all_known

    print(f"  {original_name!r} -> {corrected!r}")
    print(f"    reason: {reason}")
    print(f"    found in AusTraits? {in_austraits}")

    plantslist.loc[mask, 'name_norm'] = corrected_norm
    if in_austraits:
        plantslist.loc[mask, 'match_extended'] = True
        plantslist.loc[mask, 'match_strategy'] = 'manual_typo_fix'

  'Chysocephalum  apiculatum' -> 'Chrysocephalum apiculatum'
    reason: missing 'r' in genus
    found in AusTraits? True
  'Leptorhyncos  tenuifolius' -> 'Leptorhynchos tenuifolius'
    reason: missing 'h' in genus
    found in AusTraits? True


In [ ]:
print("\nFinal match strategy breakdown:")
print(plantslist['match_strategy'].value_counts())
print(f"\nTotal matched: {plantslist['match_extended'].sum()} / {len(plantslist)}")


Final match strategy breakdown:
match_strategy
exact_taxon               324
via_original_name          91
species_level_fallback     53
unmatched                   4
manual_typo_fix             2
Name: count, dtype: int64

Total matched: 470 / 474


In [ ]:
plantslist.to_csv('plantslist_matched.csv', index=False)

**MODULE 3 - Filter traits.csv**

File used: traits.csv

What we're doing:

- Streaming traits.csv 200K rows at a time.
- For each chunk, keeping only rows that satisfy both conditions:

  - trait_name is one of our 15 wildlife traits.
  - taxon_name (normalised) OR original_name (normalised) matches one of our 470 matched plants.

- Concatenating the kept chunks into one DataFrame called filtered_traits.
- Printing per-trait row counts so we can spot any trait that returned 0 rows (would mean a name typo in our target_traits list).

In [ ]:
#path alreday defined
target_traits = [
    #pollinators
    "pollination_syndrome" , "pollination_system" , "flower_colour" , "flowering_time" , "flower_diameter",
    #frugivores/seed dispersal
    "fruit_type","fruit_colour","fruit_fleshiness","fruiting_time","dispersal_syndrome","dispersers",
    #habitat/structure
    "plant_growth_form","plant_height","woodiness","leaf_phenology",
]


In [ ]:
#set of normalised matched plant names(lookup key for filtering)
matched_ones = set(plantslist[plantslist['match_extended']]['name_norm'])
print(f"Matched plants:{len(matched_ones)}")
print(f"target_traits:{len(target_traits)}")

Matched plants:439
target_traits:15


In [ ]:
kept_chunks = []
total_scanned = 0

for chunk in pd.read_csv(
    traits_path,
    chunksize=200_000,
    encoding='latin-1',
    low_memory=False,
    usecols=['taxon_name', 'original_name', 'trait_name', 'value', 'unit', 'value_type'],
):
    total_scanned += len(chunk)

    # Normalise both name columns
    taxon_norm = chunk['taxon_name'].fillna('').str.strip().str.lower()
    orig_norm  = chunk['original_name'].fillna('').str.strip().str.lower()

    # Two filters: trait must be relevant AND (taxon OR original) must be one of our plants
    keep_mask = (
        chunk['trait_name'].isin(target_traits) &
        (taxon_norm.isin(matched_ones) | orig_norm.isin(matched_ones))
    )

    if keep_mask.any():
        kept_chunks.append(chunk[keep_mask])

filtered_traits = (
    pd.concat(kept_chunks, ignore_index=True)
    if kept_chunks else pd.DataFrame()
)

print(f"Scanned {total_scanned:,} rows -> kept {len(filtered_traits):,}")
print(f"Unique species in filtered data: {filtered_traits['taxon_name'].nunique()}")


Scanned 1,798,215 rows -> kept 14,595
Unique species in filtered data: 427


In [ ]:
print("\nRows per trait:")
counts = filtered_traits['trait_name'].value_counts()
for t in target_traits:
    n = int(counts.get(t, 0))
    flag = "WARN" if n == 0 else "ok  "
    print(f"  [{flag}]  {t:<25} {n:>6}")


Rows per trait:
  [ok  ]  pollination_syndrome         870
  [ok  ]  pollination_system           241
  [ok  ]  flower_colour                307
  [ok  ]  flowering_time              1351
  [ok  ]  flower_diameter               11
  [ok  ]  fruit_type                  1411
  [ok  ]  fruit_colour                 270
  [ok  ]  fruit_fleshiness             677
  [ok  ]  fruiting_time                 71
  [ok  ]  dispersal_syndrome          1727
  [ok  ]  dispersers                   964
  [ok  ]  plant_growth_form           3756
  [ok  ]  plant_height                2401
  [ok  ]  woodiness                    386
  [ok  ]  leaf_phenology               152


**MODULE 4 - Aggregate duplicate observations**

using filtered_traits from module 3 ,

What we're doing:

- Step A - add a plant_key column that points each row at the right plant in plantslist. This handles the synonym case: a row's taxon_name might be the AusTraits accepted name, but our plant matched via original_name. We pick whichever one is in our matched set.
- Step B - split rows into numeric vs. categorical and aggregate each path separately:

  - flower_diameter, plant_height : median across observations.
  - All other traits : sorted unique values joined with commas (eg. "white, yellow").


- Result: one row per (plant, trait) pair.

In [ ]:
numeric_traits = {"flower_diameter" , "plant_height"}

#mapping
taxon_norm = filtered_traits['taxon_name'].fillna('').str.strip().str.lower()
orig_norm  = filtered_traits['original_name'].fillna('').str.strip().str.lower()

#prefer taxon_name match
filtered_traits['plant_key'] = taxon_norm.where(
    taxon_norm.isin(matched_ones), orig_norm
)

mapped = filtered_traits['plant_key'].isin(matched_ones).sum()
print(f"Rows mapped to a plant: {mapped} / {len(filtered_traits)}")

#dropping unmatched rows
filtered_traits = filtered_traits[filtered_traits['plant_key'].isin(matched_ones)]

Rows mapped to a plant: 14595 / 14595


In [ ]:
#aggregate numeric traits
numeric_mask = filtered_traits['trait_name'].isin(numeric_traits)
numeric_df   = filtered_traits[numeric_mask].copy()
numeric_df['value_num'] = pd.to_numeric(numeric_df['value'], errors='coerce')

numeric_agg = (
    numeric_df.dropna(subset=['value_num'])
    .groupby(['plant_key', 'trait_name'], as_index=False)['value_num']
    .median()
    .rename(columns={'value_num': 'value'})
)
print(f"\nNumeric (plant, trait) pairs: {len(numeric_agg)}")


Numeric (plant, trait) pairs: 380


In [ ]:
#aggregate categorical traits
categorical_df = filtered_traits[~numeric_mask].copy()
categorical_df['value_str'] = categorical_df['value'].astype(str).str.strip()
categorical_df = categorical_df[categorical_df['value_str'].ne('') & categorical_df['value_str'].ne('nan')]

categorical_agg = (
    categorical_df
    .groupby(['plant_key', 'trait_name'], as_index=False)['value_str']
    .agg(lambda x: ', '.join(sorted(x.unique())))
    .rename(columns={'value_str': 'value'})
)
print(f"Categorical (plant, trait) pairs: {len(categorical_agg)}")

Categorical (plant, trait) pairs: 3800


In [ ]:
#combining both
agg = pd.concat([numeric_agg, categorical_agg], ignore_index=True)
print(f"\nTotal aggregated rows: {len(agg)}")
print("\nSample (first 15 rows):")
print(agg.head(15).to_string(index=False))


Total aggregated rows: 4180

Sample (first 15 rows):
           plant_key   trait_name  value
acacia acinacea s.l. plant_height    1.5
acacia aculeatissima plant_height    0.5
     acacia dealbata plant_height   30.0
 acacia genistifolia plant_height    2.0
      acacia implexa plant_height  14.25
     acacia mearnsii plant_height   10.0
  acacia melanoxylon plant_height   30.0
      acacia montana plant_height    3.5
    acacia mucronata plant_height   5.75
   acacia myrtifolia plant_height    1.0
     acacia oswaldii plant_height    6.0
     acacia paradoxa plant_height    3.0
    acacia pycnantha plant_height    8.0
       acacia rubida plant_height    5.0
  acacia verniciflua plant_height    4.2


In [ ]:
agg.to_csv('aggregated_traits.csv', index=False)

In [ ]:
agg.shape

(4180, 3)

**MODULE 5 - Generate pivot table**

using agg from module 4

What we're doing:

- Reshape so each plant is a row and each trait is a column.
- Currently agg is "long" (3 columns: plant_key, trait_name, value); we need "wide" (16 columns: plant_key + 15 traits).
- Force all 15 target traits to exist as columns even if a trait had zero data — keeps the schema predictable for downstream consumers.

In [ ]:
wide = agg.pivot(index='plant_key', columns='trait_name', values='value').reset_index()

#Guarantee every target trait column exists, even if AusTraits had no data for it
for t in target_traits:
    if t not in wide.columns:
        wide[t] = pd.NA

# Reorder columns: plant_key first, then traits in our canonical order
wide = wide[['plant_key'] + target_traits]

In [ ]:
print(f"Wide shape: {wide.shape}")
print(f"Columns: {list(wide.columns)}")
print("\nSample (first 5 rows, key + 4 traits):")
print(wide[['plant_key', 'plant_growth_form', 'plant_height',
           'flower_colour', 'flowering_time']].head().to_string(index=False))


Wide shape: (428, 16)
Columns: ['plant_key', 'pollination_syndrome', 'pollination_system', 'flower_colour', 'flowering_time', 'flower_diameter', 'fruit_type', 'fruit_colour', 'fruit_fleshiness', 'fruiting_time', 'dispersal_syndrome', 'dispersers', 'plant_growth_form', 'plant_height', 'woodiness', 'leaf_phenology']

Sample (first 5 rows, key + 4 traits):
           plant_key       plant_growth_form plant_height             flower_colour                                                       flowering_time
acacia acinacea s.l.                   shrub          1.5                       NaN                                                         nnnnnnyyyyyn
acacia aculeatissima         shrub, subshrub          0.5             yellow_orange                                           nnnnnnnyyynn, nnnnnnnyyyyn
     acacia dealbata        shrub tree, tree         30.0             yellow_orange                             nnnnnnnyyynn, nnnnnnyyyynn, nnnnnnyyyyyn
 acacia genistifolia       shrub

In [ ]:
wide.to_csv('traits_wide.csv', index=False)

**MODULE 6 - Left-join and merge**

What we're doing:

- Left-join plantslist -  wide on the name_norm column.Every plant in plantslist survives, even ones with no AusTraits data (they just get NaN trait columns).

- Drop helper columns (plant_key, name_norm, match_taxon)

- Keep match_strategy as an audit column so anyone reading the dataset knows how each plant was matched (or that it wasn't).

- Generate a coverage report showing % of plants with data per trait.

In [ ]:
#left join
final = plantslist.merge(wide, how='left',left_on='name_norm',right_on='plant_key')

In [ ]:
#preserving columns we need for the final o/p
keep_cols = ['id','name','commonname','lf','match_strategy'] + target_traits
final = final[keep_cols]

In [ ]:
print(f"Final shape: {final.shape}")
print(f"Columns: {list(final.columns)}")

Final shape: (474, 20)
Columns: ['id', 'name', 'commonname', 'lf', 'match_strategy', 'pollination_syndrome', 'pollination_system', 'flower_colour', 'flowering_time', 'flower_diameter', 'fruit_type', 'fruit_colour', 'fruit_fleshiness', 'fruiting_time', 'dispersal_syndrome', 'dispersers', 'plant_growth_form', 'plant_height', 'woodiness', 'leaf_phenology']


In [ ]:
#coverage report
coverage_rows = []
for t in target_traits:
  n = int(final[t].notna().sum())
  pct = round(n/len(final)*100 , 1)
  coverage_rows.append({'tarit':t , 'n_with_data':n, 'pct_coverage':pct})
  print(f"{t:25} {n:>4} ({pct:>5}%)")

pollination_syndrome       428 ( 90.3%)
pollination_system         261 ( 55.1%)
flower_colour              262 ( 55.3%)
flowering_time             426 ( 89.9%)
flower_diameter              6 (  1.3%)
fruit_type                 379 ( 80.0%)
fruit_colour               150 ( 31.6%)
fruit_fleshiness           365 ( 77.0%)
fruiting_time               65 ( 13.7%)
dispersal_syndrome         457 ( 96.4%)
dispersers                 423 ( 89.2%)
plant_growth_form          451 ( 95.1%)
plant_height               405 ( 85.4%)
woodiness                  291 ( 61.4%)
leaf_phenology             110 ( 23.2%)


In [ ]:
coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv('coverage_report.csv', index=False)

In [ ]:
final.to_csv('plants_enriched_final.csv', index=False)

In [ ]:
final.shape

(474, 20)

In [ ]:
final.head().to_string(index = False)

' id                 name        commonname lf    match_strategy pollination_syndrome   pollination_system             flower_colour                                                       flowering_time flower_diameter fruit_type                                  fruit_colour fruit_fleshiness fruiting_time                                          dispersal_syndrome                              dispersers       plant_growth_form plant_height woodiness leaf_phenology\n  1 Acacia acinacea s.l.  Gold-dust Wattle MS via_original_name               insect                  NaN                       NaN                                                         nnnnnnyyyyyn             NaN        NaN                                           NaN              NaN           NaN                                      myrmecochory, zoochory                           invertebrates                   shrub          1.5       NaN            NaN\n  2 Acacia aculeatissima Thin-leaf  Wattle SS       exact_taxon